# Clase 14 — Avellaneda-Stoikov II — Simulación

Poner el modelo a correr: simular el market maker A-S contra un mid que se mueve, ver cómo controla el inventario, y barrer gamma para entender el trade-off riesgo/PnL.

**Hoy construyes:** simular A-S y barrer parámetros.

## Cómo usar este cuaderno

- **Núcleo (en clase):** ejercicios 1 a 3.
- **Si vamos bien:** ejercicios 4 en adelante.
- **Casa / auxiliares:** el cuaderno `*_auxiliary.ipynb`.

Inténtalo, ejecuta la comprobación (`assert`) y mira la solución solo si te atascas.

## 1. Simula el A-S

**Practicas:** MMSimulation con A-S.

Simula `AvellanedaStoikov('BTC', gamma=0.1, sigma=0.5, horizon=400)` con `steps=400`. Guarda `max_inv` y `pnl`.

In [ ]:
from exchange.strategies import AvellanedaStoikov
from exchange.simulation import MMSimulation
max_inv = None
pnl = None

In [ ]:
assert max_inv >= 0 and isinstance(pnl, float)
print('ok  max|inv|=%.2f pnl=%.2f' % (max_inv, pnl))

### Solución guiada

```python
res = MMSimulation(AvellanedaStoikov('BTC', gamma=0.1, sigma=0.5, horizon=400), steps=400).run()
max_inv = res.max_inventory
pnl = res.final_pnl
```

## 2. El skew reduce el inventario

**Practicas:** comparar con / sin skew.

Con la misma semilla, simula un MarketMaker con skew (2.0) y otro sin skew (0.0), half_spread 0.3. Guarda `inv_skew` e `inv_noskew` (max inventario). El skew debe dejar menos inventario.

In [ ]:
from exchange.strategies import MarketMaker
from exchange.simulation import MMSimulation
inv_skew = None
inv_noskew = None

In [ ]:
assert inv_skew <= inv_noskew + 1e-9, 'el skew empuja el inventario hacia 0'
print('ok  skew=%.3f noskew=%.3f' % (inv_skew, inv_noskew))

### Solución guiada

```python
inv_skew = MMSimulation(MarketMaker('BTC', half_spread=0.3, inventory_skew=2.0), steps=400).run().max_inventory
inv_noskew = MMSimulation(MarketMaker('BTC', half_spread=0.3, inventory_skew=0.0), steps=400).run().max_inventory
```

## 3. Más gamma, más inclina el reservation price

**Practicas:** trade-off riesgo/PnL.

Con inventario fijo (5), mide cuánto se aleja el reservation price del mid para gamma 0.05 y 0.8. Guarda `skew_low_gamma` y `skew_high_gamma` (= |r - mid|). Más gamma -> más inclinación -> antes sueltas inventario.

In [ ]:
from exchange.strategies import AvellanedaStoikov
def skew_mag(g):
    s = AvellanedaStoikov('BTC', gamma=g, sigma=10, kappa=1.5, horizon=500)
    s._inventory = 5; s._t = 0
    return abs(s.reservation_price(100) - 100)
skew_low_gamma = None
skew_high_gamma = None

In [ ]:
assert skew_high_gamma > skew_low_gamma, 'más gamma = más aversión = más inclinación'
print('ok  g=0.05 -> %.1f | g=0.8 -> %.1f' % (skew_low_gamma, skew_high_gamma))

### Solución guiada

```python
skew_low_gamma = skew_mag(0.05)
skew_high_gamma = skew_mag(0.8)
```

## Cierre

Más gamma = más miedo al inventario: cotizas más defensivo, cargas menos posición, pero capturas menos spread.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.